# COMP 472: Artificial Intelligence
## Lab: Neural Networks

### What is a neural network?

A neural network is a model built from many small, simple units called **neurons**, connected in **layers**. Each neuron receives a few numbers, multiplies each one by a **weight**, adds them up, and passes the result through a simple function. The output of one layer becomes the input of the next.

* The **input layer** is just the data (pixel values, sensor readings, word features, ...).
* One or more **hidden layers** transform the data step by step. They are called hidden because we never see or label their values directly.
* The **output layer** gives the answer: a class, a probability, a number.

The weights are not written by a programmer. They are **learned from examples**: we show the network inputs together with the correct answers, measure how wrong it is, and adjust the weights a little to make it less wrong. Repeat this thousands of times and the network gets good at the task.

The name comes from biology. A brain neuron collects electrical signals from other neurons and "fires" when the total signal is strong enough. The artificial neuron is a heavy simplification of that idea (McCulloch and Pitts, 1943; Rosenblatt, 1958). Modern networks owe more to linear algebra and calculus than to neuroscience.

### Why do we need them?

Many tasks have no rules we can write down. Try writing an `if` statement that decides whether an $8\times8$ image shows a 3 or an 8, or whether a sentence is sarcastic. Classical methods help, but have limits:

* **Linear models** (linear and logistic regression) can only draw straight lines (or flat planes) between classes.
* Most classical methods work well only after a human designs good **features** for them (edges, word counts, ratios, ...). This feature engineering is slow and needs domain knowledge.

A neural network with hidden layers can learn curved, complicated decision boundaries, and it learns useful features from raw data on its own. The **universal approximation theorem** (Cybenko, 1989; Hornik, 1991) says that a network with a single hidden layer and enough neurons can approximate any continuous function on a bounded input range as closely as we like. Note what it does *not* say: it does not tell us how many neurons we need, or that training will actually find the right weights.

### Advantages

* **Non-linear:** they can model relationships that straight lines cannot.
* **Learn their own features** (representation learning): early layers learn simple patterns, later layers combine them into more abstract ones.
* **Get better with more data:** many classical models stop improving after a point; large networks keep improving.
* **One framework for many kinds of data:** images, sound, text, tables, games. Only the architecture changes.
* **Fast on modern hardware:** the core computation is matrix multiplication, which GPUs do extremely well.
* **Reusable:** a network trained on one task can be fine-tuned for a related task with little data (transfer learning).

### Limitations

* They usually need **a lot of labelled data**. On small tabular datasets, a decision tree or logistic regression is often just as good.
* Training can be **computationally expensive**.
* They are **hard to interpret**. We can see millions of weights, but not easily *why* the network made a decision.
* There are many **hyperparameters** (layers, units, learning rate, ...) to choose.
* They can be **confidently wrong**, and small, carefully chosen changes to the input can fool them.

### Where are they used?

Image recognition, speech recognition, machine translation, recommendation systems, medical image analysis, self-driving perception, and the large language models behind chatbots. All of them are built from the same basic pieces you will write today.

**What you need:** `numpy`, `matplotlib`, `scikit-learn`, `torch`. `keras` is optional. Google Colab already has all of them.

## Outline

**0. Setup**

**1. The single neuron**
- 1.1 What one neuron computes
- 1.2 Activation functions
- 1.3 A neuron as a logic gate
- 1.4 The perceptron learning rule
- 1.5 Where a single neuron fails: XOR

**2. A small network (2 inputs, 2 hidden units, 1 output)**
- 2.1 Layers are matrix multiplications
- 2.2 Some vocabulary
- 2.3 Solving XOR with hand-picked weights

**3. How a network learns: loss and gradient descent**
- 3.1 Loss functions: binary cross-entropy and mean squared error
- 3.2 Gradient descent on a single parameter

**4. Backpropagation from scratch**
- 4.1 The chain rule through a 2-layer network
- 4.2 Implementing forward and backward passes
- 4.3 Gradient checking
- 4.4 Training on XOR

**5. A bigger network from scratch**
- 5.1 Softmax and cross-entropy for more than two classes
- 5.2 A general `MLP` class (any number of layers)
- 5.3 A dataset made from scratch: three spirals
- 5.4 Linear model vs. neural network
- 5.5 What the hidden layer size changes

**6. A real dataset: handwritten digits**
- 6.1 Loading and looking at the data
- 6.2 Train / validation / test split and scaling
- 6.3 Training our from-scratch MLP
- 6.4 Evaluation: accuracy, confusion matrix, precision and recall

**7. Neural networks from libraries**
- 7.1 scikit-learn: `MLPClassifier`
- 7.2 PyTorch: tensors, autograd and a training loop
- 7.3 Keras: `Sequential`, `compile` and `fit`
- 7.4 Comparing all of them

**8. Practical notes**
- 8.1 The learning rate
- 8.2 Overfitting
- 8.3 Other things worth knowing

**9. Preview of next week: Convolutional Neural Networks**

**10. Further reading**

**11. Tasks**
- Task 1: tanh and leaky ReLU (plus their derivatives)
- Task 2: forward pass of a 3-4-2 network
- Task 3: mean squared error and its gradient
- Task 4: backprop with a tanh hidden layer
- Task 5: count the parameters of a network
- Task 6: build a "rings" dataset and train a network on it
- Task 7: your own PyTorch network on the digits

Sections 0 to 9 are the lesson: run them top to bottom and read along. The tasks in Section 11 use functions defined in the lesson, so run the lesson first.

## 0. Setup

* **NumPy** gives us fast arrays and linear algebra. Everything in Sections 1 to 6 is built with it.
* **Matplotlib** draws the plots.
* A **random seed** fixes the sequence of "random" numbers the computer produces. Neural networks use randomness (initial weights, shuffling the data), so without a seed you would get slightly different numbers every time you run the notebook. With a seed, your results match your neighbour's.

If you are on your own machine and something is missing, uncomment the `pip` line.

In [ ]:
# !pip install numpy matplotlib scikit-learn torch keras

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)   # nicer printing of arrays
plt.rcParams["figure.figsize"] = (6, 4)

SEED = 472                         # fixed seed so everyone gets the same results
rng = np.random.default_rng(SEED)  # NumPy's random number generator

print("numpy", np.__version__)

A small plotting helper we will reuse many times. It asks the model for a prediction at every point of a fine 2D grid and colours the grid by the predicted class. The result is a picture of the model's **decision boundary**: the border where the prediction switches from one class to another.

In [ ]:
def plot_decision_boundary(predict_fn, X, y, ax=None, title="", steps=250):
    # predict_fn takes an (n, 2) array and returns n class labels
    ax = ax if ax is not None else plt.gca()
    pad = 0.5
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    # grid of points covering the data
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, steps), np.linspace(y_min, y_max, steps))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = np.asarray(predict_fn(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="viridis")                    # coloured regions
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", edgecolors="k", s=25)  # data points
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

## 1. The single neuron

### 1.1 What one neuron computes

A neuron gets an input vector $\mathbf{x} = (x_1, \dots, x_d)$. It has one **weight** per input, $\mathbf{w} = (w_1, \dots, w_d)$, and one **bias** $b$. It does two things:

1. Computes a weighted sum, called the **pre-activation**:
$$z = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b = \mathbf{w}\cdot\mathbf{x} + b$$
2. Passes $z$ through an **activation function** $g$ to get the neuron's output, the **activation**:
$$a = g(z)$$

What each part is for:

* **Weights** say how much each input matters. A large positive weight means "this input pushes the output up", a negative weight pushes it down, a weight near 0 means "ignore this input".
* **Bias** shifts the threshold. Without it, when all inputs are 0 the sum is always 0, and the decision line is forced to pass through the origin. The bias lets the neuron move that line anywhere.
* **Activation function** decides what comes out (more in 1.2).

The oldest activation is the **step function**: output 1 if $z \ge 0$, else 0. A neuron with a step activation is called a **perceptron** (Rosenblatt, 1958).

In [ ]:
def step(z):
    # 1 where z >= 0, else 0
    return (np.asarray(z) >= 0).astype(float)

def neuron(x, w, b, activation):
    z = np.dot(w, x) + b     # weighted sum + bias
    return activation(z)     # activation function

x = np.array([2.0, 3.0])     # inputs
w = np.array([0.5, -1.0])    # weights
b = 1.0                      # bias

z = np.dot(w, x) + b
print("z =", z, "  (0.5*2 + (-1)*3 + 1)")
print("output =", neuron(x, w, b, step))

### 1.2 Activation functions

The step function has a big problem: its derivative is 0 everywhere (and undefined at 0). As we will see in Section 3, learning works by following derivatives. A derivative of 0 tells the network nothing about which way to change the weights. So in practice we use smooth or piecewise-linear functions:

| Name | Formula | Derivative | Output range |
|---|---|---|---|
| Sigmoid | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | $(0, 1)$ |
| tanh | $\dfrac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$ | $1-\tanh^2(z)$ | $(-1, 1)$ |
| ReLU | $\max(0, z)$ | $1$ if $z>0$, else $0$ | $[0, \infty)$ |
| Leaky ReLU | $z$ if $z>0$, else $\alpha z$ | $1$ if $z>0$, else $\alpha$ | $(-\infty, \infty)$ |

**Sigmoid.** Squashes any number into $(0, 1)$, so it is a natural choice for an output that should be a probability. Limitations: for large $|z|$ the curve is flat (it **saturates**), so the derivative is almost 0 and learning slows down. Its largest derivative is only 0.25, and in a deep network these small numbers get multiplied layer after layer, which makes the gradient shrink towards 0 (the **vanishing gradient** problem). Its outputs are also always positive, which makes optimisation a little less efficient.

**tanh.** Same S-shape, but centred at 0 with range $(-1, 1)$. Usually trains a bit better than sigmoid in hidden layers. It still saturates.

**ReLU** (Rectified Linear Unit). The default for hidden layers today. It is very cheap to compute, and for $z > 0$ its derivative is exactly 1, so gradients do not vanish there. Limitation: if a unit's input is negative for every example, its output and gradient are always 0 and it stops learning for good (a **dead ReLU**).

**Leaky ReLU.** Fixes dead units by giving negative inputs a small slope $\alpha$ (for example 0.01) instead of 0.

**Why do we need an activation at all?** Without one, every layer is a linear function, and a linear function of a linear function is still linear. A 10-layer network with no activations is exactly as powerful as one layer. The non-linearity is what makes adding layers useful.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)   # 1 for positive inputs, 0 otherwise

z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, f, df, name in [(axes[0], step, None, "step"),
                        (axes[1], sigmoid, sigmoid_grad, "sigmoid"),
                        (axes[2], relu, relu_grad, "ReLU")]:
    ax.plot(z, f(z), label=name)
    if df is not None:
        ax.plot(z, df(z), "--", label="derivative")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()

print("sigmoid(0)      =", sigmoid(0))
print("sigmoid'(0)     =", sigmoid_grad(0), " <- the largest slope sigmoid ever has")
print("sigmoid'(10)    =", round(sigmoid_grad(10), 6), " <- saturated, almost no gradient")
print("relu([-2,0,3])  =", relu(np.array([-2., 0., 3.])))

### 1.3 A neuron as a logic gate

With the right weights, one perceptron can compute AND, OR or NAND on two binary inputs. For AND we want the output to be 1 only when both inputs are 1. Take $w = (1, 1)$ and $b = -1.5$: the sum $x_1 + x_2 - 1.5$ is positive only for $(1,1)$.

Geometrically, the neuron draws a straight line $w_1x_1 + w_2x_2 + b = 0$ and outputs 1 on one side and 0 on the other. In more dimensions this line becomes a flat **hyperplane**. Either way, one neuron can only ever separate classes with something flat.

In [ ]:
# All four combinations of two binary inputs
X_bool = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

gates = {                                   # (weights, bias)
    "AND":  (np.array([1.0, 1.0]),   -1.5),
    "OR":   (np.array([1.0, 1.0]),   -0.5),
    "NAND": (np.array([-1.0, -1.0]),  1.5),
}
print("inputs:", X_bool.astype(int).tolist())
for name, (w, b) in gates.items():
    out = step(X_bool @ w + b)              # all 4 inputs at once
    print(f"{name:5s} -> {out}")

### 1.4 The perceptron learning rule

Picking weights by hand does not scale. Rosenblatt's rule learns them from examples. For each training example $(\mathbf{x}, y)$:

$$\hat y = \text{step}(\mathbf{w}\cdot\mathbf{x} + b)$$
$$\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat y)\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - \hat y)$$

If the prediction is right, $y - \hat y = 0$ and nothing changes. If it predicted 0 but the answer is 1, the weights move towards $\mathbf{x}$ so that next time the sum is larger; if it predicted 1 but the answer is 0, they move away.

Two terms you will see everywhere:

* **Learning rate** $\eta$: how big each correction is. Too small and learning is slow; too big and it overshoots.
* **Epoch**: one full pass over the whole training set.

The **perceptron convergence theorem** says: if some straight line separates the two classes perfectly, this rule finds one in a finite number of steps.

**Limitations.** If no such line exists, the rule never settles; it keeps changing the weights forever. And because the step function has no useful derivative, this rule cannot be extended to train hidden layers. That is why modern networks use smooth activations and gradient descent instead (Sections 3 and 4).

In [ ]:
def train_perceptron(X, y, lr=0.1, epochs=20):
    w = np.zeros(X.shape[1])    # start with all weights at 0
    b = 0.0
    errors_per_epoch = []
    for epoch in range(epochs):
        errors = 0
        for xi, target in zip(X, y):          # one example at a time
            pred = step(np.dot(w, xi) + b)
            update = lr * (target - pred)     # 0 if the prediction was right
            w += update * xi
            b += update
            errors += int(update != 0)
        errors_per_epoch.append(errors)
    return w, b, errors_per_epoch

y_and = np.array([0, 0, 0, 1])
w_and, b_and, err_and = train_perceptron(X_bool, y_and)
print("learned w =", w_and, " b =", round(b_and, 3))
print("mistakes per epoch:", err_and)
print("predictions:", step(X_bool @ w_and + b_and), " target:", y_and)

### 1.5 Where a single neuron fails: XOR

XOR outputs 1 when exactly one input is 1. Plot the four points and try to draw one straight line with the two yellow points on one side and the two purple points on the other. You cannot.

A dataset is **linearly separable** if a single straight line (hyperplane) can split the classes. XOR is not, and so no single perceptron can learn it, however long we train. Watch the number of mistakes: it never reaches 0.

This observation (Minsky and Papert, *Perceptrons*, 1969) was one of the reasons interest in neural networks dropped for over a decade. The fix is to add a hidden layer, which is what Section 2 is about.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
w_x, b_x, err_xor = train_perceptron(X_bool, y_xor, epochs=50)
print("mistakes per epoch (last 10):", err_xor[-10:])   # never gets to 0

plt.figure(figsize=(3.5, 3.5))
plt.scatter(X_bool[:, 0], X_bool[:, 1], c=y_xor, s=200, cmap="viridis", edgecolors="k")
plt.title("XOR: no single line separates the colours")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.show()

## 2. A small network (2 inputs, 2 hidden units, 1 output)

### 2.1 Layers are matrix multiplications

Instead of looping over neurons one by one, we put all the weights of a layer in a matrix. We use the convention that **each row of `X` is one example**:

* `X` has shape `(n, d_in)`: $n$ examples, $d_{in}$ features each
* `W` has shape `(d_in, d_out)`: column $j$ holds the weights of neuron $j$
* `b` has shape `(d_out,)`: one bias per neuron

Then a whole layer, for all examples at once, is

$$Z = XW + b, \qquad A = g(Z)$$

and `Z` and `A` have shape `(n, d_out)`.

**Broadcasting.** `XW` has shape `(n, d_out)` but `b` has shape `(d_out,)`. NumPy automatically adds `b` to every row. This rule is called broadcasting, and it saves us from writing loops.

**Why bother with matrices?** It is much shorter to write, and much faster: NumPy (and GPUs) run matrix multiplication with highly optimised code, while Python loops are slow. The cell below computes the same layer both ways and times them.

> Getting shapes right is most of the work when debugging neural network code. When something breaks, print `.shape` of everything.

In [ ]:
import time

n, d_in, d_out = 2000, 30, 40
X_demo = rng.normal(size=(n, d_in))
W_demo = rng.normal(size=(d_in, d_out))
b_demo = np.zeros(d_out)

t0 = time.perf_counter()
Z_demo = X_demo @ W_demo + b_demo                  # one line for the whole layer
t_matrix = time.perf_counter() - t0

t0 = time.perf_counter()
Z_loop = np.array([[np.dot(W_demo[:, j], X_demo[i]) + b_demo[j]   # neuron by neuron
                    for j in range(d_out)] for i in range(n)])
t_loop = time.perf_counter() - t0

print("X:", X_demo.shape, " W:", W_demo.shape, " b:", b_demo.shape, " ->  Z:", Z_demo.shape)
print("same result:", np.allclose(Z_demo, Z_loop))
print(f"matrix version: {t_matrix*1000:.2f} ms   loop version: {t_loop*1000:.1f} ms")

### 2.2 Some concepts

* **Multilayer perceptron (MLP)** or **feedforward network**: layers of neurons where information flows in one direction, from input to output. This is the kind of network we build in this lab.
* **Fully connected** (or **dense**) layer: every neuron is connected to every output of the previous layer. That is exactly one matrix multiplication.
* **Width**: number of neurons in a layer. **Depth**: number of layers. "Deep learning" simply means networks with many layers.
* **Parameters**: the numbers that are learned (all weights and biases).
* **Hyperparameters**: the numbers *we* choose before training (number of layers, units per layer, learning rate, number of epochs, ...).
* **Forward pass**: computing the output from the input, layer by layer:
$$H = g_1(XW_1 + b_1), \qquad \hat Y = g_2(HW_2 + b_2)$$

### 2.3 Solving XOR with hand-picked weights

Note that $\text{XOR}(x_1, x_2) = \text{AND}\big(\text{OR}(x_1,x_2),\ \text{NAND}(x_1,x_2)\big)$. We already have all three gates from Section 1.3. So:

* hidden unit 1 computes OR
* hidden unit 2 computes NAND
* the output unit computes AND of the two hidden units

Look at the second plot. The hidden layer has moved the four points into a new space (its **hidden representation**) where a straight line *does* separate the classes. This is the central idea of hidden layers: each layer re-describes the data so that the next layer's job becomes easier. In Section 4 the network will find such a representation by itself.

In [ ]:
W1 = np.array([[1.0, -1.0],      # column 0 = OR unit, column 1 = NAND unit
               [1.0, -1.0]])
b1 = np.array([-0.5, 1.5])
W2 = np.array([[1.0],
               [1.0]])           # output = AND of the two hidden units
b2 = np.array([-1.5])

H = step(X_bool @ W1 + b1)           # hidden layer
y_hat = step(H @ W2 + b2).ravel()    # output layer

print("hidden layer output (OR, NAND):\n", H)
print("network output:", y_hat, "   XOR target:", y_xor)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(X_bool[:, 0], X_bool[:, 1], c=y_xor, s=200, cmap="viridis", edgecolors="k")
axes[0].set_title("input space"); axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("$x_2$")
# small jitter so overlapping points are visible
axes[1].scatter(H[:, 0] + rng.normal(0, 0.03, 4), H[:, 1] + rng.normal(0, 0.03, 4),
                c=y_xor, s=200, cmap="viridis", edgecolors="k")
axes[1].plot([0.2, 1.3], [1.3, 0.2], "r--")
axes[1].set_title("hidden space: now a line works"); axes[1].set_xlabel("OR"); axes[1].set_ylabel("NAND")
plt.tight_layout(); plt.show()

## 3. How a network learns: loss and gradient descent

In Section 2 we chose the weights ourselves. A real network has thousands or millions of weights, so we need a way to *learn* them from data. The recipe is always the same:

1. **Loss function**: a single number that measures how wrong the network is on the training data. Training means making this number small.
2. **Gradient**: the vector of derivatives of the loss with respect to every weight. Each entry says "if this weight goes up a little, the loss changes this much". The gradient points in the direction where the loss *increases* fastest.
3. **Gradient descent**: move every weight a small step in the *opposite* direction of its gradient. Repeat.

### 3.1 Loss functions

**Binary cross-entropy (BCE)** is used for two-class problems where the network outputs a probability $\hat y \in (0,1)$ (sigmoid output):

$$L_{BCE} = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \log \hat y_i + (1-y_i)\log(1-\hat y_i)\Big]$$

When $y=1$ only the first term counts and the loss is $-\log \hat y$: close to 0 when $\hat y$ is near 1, and huge when $\hat y$ is near 0. So BCE punishes confident wrong answers very strongly. It also has a statistical meaning: minimising BCE is the same as maximising the likelihood of the labels under the model.

**Mean squared error (MSE)** is used for **regression** (predicting numbers):

$$L_{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat y_i - y_i)^2$$

Squaring makes all errors positive and punishes big errors more than small ones. Limitation: one outlier with a huge error can dominate the whole loss.

**Why not use MSE for classification?** You can, but combined with a sigmoid output it trains badly: when the network is very wrong, the sigmoid is saturated and the MSE gradient is tiny, so the network learns slowest exactly when it should learn fastest. Cross-entropy does not have this problem (you will see the nice gradient it gives in Section 4.1).

In the code we **clip** predictions slightly away from 0 and 1, because $\log(0) = -\infty$ would break the computation.

In [ ]:
def bce(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1 - eps)      # avoid log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

y_true = np.array([1, 0, 1])
print("good predictions    :", round(bce(y_true, np.array([0.9, 0.1, 0.8])), 4))
print("unsure (all 0.5)    :", round(bce(y_true, np.array([0.5, 0.5, 0.5])), 4), " = log(2)")
print("confident but wrong :", round(bce(y_true, np.array([0.01, 0.99, 0.01])), 4))

### 3.2 Gradient descent on a single parameter

Before doing this for a whole network, look at the simplest case. Take $f(w) = (w-3)^2$. Its minimum is at $w=3$ and its derivative is $f'(w) = 2(w-3)$. Gradient descent repeats

$$w \leftarrow w - \eta\, f'(w)$$

Below we try three learning rates. Too small and it crawls. Well chosen and it reaches the minimum quickly. Too big and every step jumps over the minimum to a point that is even worse, so it **diverges**. Exactly the same happens when training a real network with a badly chosen learning rate.

**Limitation.** This function has a single minimum, so gradient descent always finds it. The loss of a neural network is not like that: it has many **local minima**, flat regions and **saddle points**. Gradient descent only guarantees that we go downhill, not that we reach the lowest point. In practice, for large networks most of the minima it finds are good enough.

In [ ]:
def gradient_descent_1d(lr, w0=-2.0, steps=25):
    ws = [w0]
    for _ in range(steps):
        w = ws[-1]
        ws.append(w - lr * 2 * (w - 3))   # w <- w - lr * f'(w)
    return np.array(ws)

w_axis = np.linspace(-3, 9, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, lr in zip(axes, [0.02, 0.3, 1.05]):
    ws = gradient_descent_1d(lr)
    ax.plot(w_axis, (w_axis - 3) ** 2, color="gray")
    ax.plot(ws, (ws - 3) ** 2, "o-", ms=4)
    ax.set_title(f"lr = {lr},  final w = {ws[-1]:.2f}")
    ax.set_ylim(-2, 60)
plt.tight_layout(); plt.show()

## 4. Backpropagation from scratch

To use gradient descent we need the derivative of the loss with respect to **every** weight in the network. **Backpropagation** (Rumelhart, Hinton and Williams, 1986) is the algorithm that computes all of them efficiently.

The idea: the network is a chain of simple operations (multiply, add, activation, loss). The chain rule of calculus says the derivative of a chain is the product of the derivatives of its pieces. Backprop starts at the loss and walks **backwards** through the network, layer by layer, reusing the values saved during the forward pass and the derivatives already computed for the layer after. That way, getting all the gradients costs about as much as one extra forward pass, instead of one pass per weight.

### 4.1 The chain rule through a 2-layer network

Our network for binary classification:

$$Z_1 = XW_1 + b_1,\quad A_1 = \sigma(Z_1),\quad Z_2 = A_1W_2 + b_2,\quad A_2 = \sigma(Z_2) = \hat Y,\quad L = \text{BCE}(Y, A_2)$$

We need $\partial L/\partial W_1$, $\partial L/\partial b_1$, $\partial L/\partial W_2$, $\partial L/\partial b_2$. We write $dX$ as short for $\partial L / \partial X$.

**Output layer.** With sigmoid + BCE something nice happens: the derivative of the loss with respect to $Z_2$ simplifies to

$$dZ_2 = \frac{1}{N}(A_2 - Y)$$

(The $\sigma(1-\sigma)$ from the sigmoid cancels against the $\frac{1}{\hat y(1-\hat y)}$ from the log. Worth checking on paper once.) The gradient is just "prediction minus target": large when the network is wrong, small when it is right.

Then, since $Z_2 = A_1W_2 + b_2$:

$$dW_2 = A_1^\top dZ_2, \qquad db_2 = \sum_{\text{rows}} dZ_2, \qquad dA_1 = dZ_2\, W_2^\top$$

**Hidden layer.** Go back through the activation (element-wise product $\odot$), then the linear part again:

$$dZ_1 = dA_1 \odot \sigma'(Z_1) = dA_1 \odot A_1(1-A_1)$$
$$dW_1 = X^\top dZ_1, \qquad db_1 = \sum_{\text{rows}} dZ_1$$

The term $\sigma'(Z_1)$ is where the **vanishing gradient** comes from: it is at most 0.25, and in a deep network the gradient is multiplied by one such factor per layer, so early layers receive tiny gradients. This is a big reason why ReLU replaced sigmoid in hidden layers.

A quick sanity rule: **the gradient of a parameter always has the same shape as the parameter**. $dW_2 = A_1^\top dZ_2$ has shape $(h, n)\times(n, 1) = (h, 1)$, the same as $W_2$.

### 4.2 Implementing forward and backward passes

We store the parameters in a dictionary and keep the intermediate values of the forward pass in a `cache`, because the backward pass needs them.

Weights start as small random numbers. They cannot all start at 0: then every hidden unit computes the same thing, receives the same gradient, and they stay identical forever (the **symmetry** problem). Biases can start at 0.

In [ ]:
def init_params(n_in, n_hidden, n_out, seed=0):
    r = np.random.default_rng(seed)
    return {
        "W1": r.normal(0, 1, size=(n_in, n_hidden)),   # random, to break symmetry
        "b1": np.zeros(n_hidden),
        "W2": r.normal(0, 1, size=(n_hidden, n_out)),
        "b2": np.zeros(n_out),
    }

def forward(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = sigmoid(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = sigmoid(Z2)
    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}   # saved for backward
    return A2, cache

def backward(params, cache, Y):
    N = Y.shape[0]
    X, A1, A2 = cache["X"], cache["A1"], cache["A2"]

    # output layer
    dZ2 = (A2 - Y) / N
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)

    # hidden layer
    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * A1 * (1 - A1)          # sigmoid derivative
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

params = init_params(2, 3, 1)
Y_xor = y_xor.reshape(-1, 1).astype(float)   # column vector, shape (4, 1)
A2, cache = forward(params, X_bool)
grads = backward(params, cache, Y_xor)

for k in params:
    print(f"{k}: param {params[k].shape}  grad {grads[k].shape}")

### 4.3 Gradient checking

How do we know `backward` is correct? A bug in backprop often does not crash anything; the network just learns badly, and you never find out why. So we compare it with a **numerical gradient**, using the definition of the derivative. For any parameter $\theta$:

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$$

We then compute the **relative error** between the two: $\frac{|a - b|}{|a| + |b|}$. Below about $10^{-6}$ means the backprop is almost certainly right; around $10^{-2}$ or larger means there is a bug.

**Limitations.** This needs two forward passes *per weight*, so it is far too slow to use for training. Use it only on a tiny network to test your code. It can also give false alarms at points where the function has a kink (like ReLU at 0), and the result depends on the choice of $\epsilon$.

In [ ]:
def numerical_grads(loss_of_params, params, eps=1e-5):
    num = {}
    for name, p in params.items():
        g = np.zeros_like(p)
        for idx in np.ndindex(p.shape):        # every single entry
            old = p[idx]
            p[idx] = old + eps; plus = loss_of_params()
            p[idx] = old - eps; minus = loss_of_params()
            p[idx] = old                       # put it back
            g[idx] = (plus - minus) / (2 * eps)
        num[name] = g
    return num

def rel_error(a, b):
    return np.max(np.abs(a - b) / np.maximum(1e-8, np.abs(a) + np.abs(b)))

params = init_params(2, 3, 1, seed=1)
A2, cache = forward(params, X_bool)
analytic = backward(params, cache, Y_xor)
numeric = numerical_grads(lambda: bce(Y_xor, forward(params, X_bool)[0]), params)

for k in params:
    print(f"{k}: relative error = {rel_error(analytic[k], numeric[k]):.2e}")

### 4.4 Training on XOR

Now we put it together: forward, loss, backward, update, repeat. This loop is the skeleton of **every** neural network training procedure, including the ones hidden inside PyTorch and Keras.

Here every update uses all four examples at once. This is called **(full) batch gradient descent**. In Section 5 we switch to mini-batches.

We use 4 hidden units. With only 2 the network *can* represent XOR (we did it by hand), but gradient descent sometimes gets stuck in a poor local minimum. A few extra units make training much more reliable. This is a general pattern: slightly larger networks are often easier to train. After the lab, try `n_hidden=2` with different seeds.

In [ ]:
def train_xor(n_hidden=4, lr=1.0, epochs=5000, seed=0):
    params = init_params(2, n_hidden, 1, seed=seed)
    losses = []
    for epoch in range(epochs):
        A2, cache = forward(params, X_bool)          # 1. forward
        losses.append(bce(Y_xor, A2))                # 2. loss
        grads = backward(params, cache, Y_xor)       # 3. backward
        for k in params:                             # 4. update
            params[k] -= lr * grads[k]
    return params, losses

xor_params, xor_losses = train_xor()
xor_prob = forward(xor_params, X_bool)[0].ravel()
print("probabilities:", xor_prob.round(3))
print("predictions  :", (xor_prob > 0.5).astype(int), "  target:", y_xor)
print("final loss   :", round(xor_losses[-1], 4))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(xor_losses); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].set_title("training loss")
plot_decision_boundary(lambda G: (forward(xor_params, G)[0] > 0.5).astype(int), X_bool, y_xor,
                       ax=axes[1], title="learned decision regions")
plt.tight_layout(); plt.show()

## 5. A bigger network from scratch

So far: one hidden layer, one output, four data points. Now we generalise to

* **any number of layers** of any size,
* **any number of classes**,
* **mini-batches** of data instead of the whole dataset at every step.

### 5.1 Softmax and cross-entropy for more than two classes

With $K$ classes the output layer has $K$ units, one score per class. These raw scores are called **logits**. **Softmax** turns them into probabilities that are positive and sum to 1:

$$\text{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

The exponential makes everything positive and exaggerates differences, so the largest logit gets most of the probability. With $K=2$ softmax is equivalent to a sigmoid.

**One-hot encoding.** Labels like 0, 1, 2 are names, not quantities: class 2 is not "twice" class 1. So we write each label as a vector with a 1 in the position of its class and 0 elsewhere. Class 2 out of 4 becomes `[0, 0, 1, 0]`. This vector is the "target probability distribution".

The loss is **categorical cross-entropy**, where $Y$ is one-hot:

$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} Y_{ik}\log P_{ik}$$

Since only one entry of each row of $Y$ is 1, this is just the average of $-\log(\text{probability given to the correct class})$. And just like sigmoid + BCE, the gradient at the output is simply

$$dZ = \frac{1}{N}(P - Y)$$

**Numerical stability.** $e^{1000}$ overflows to `inf`, and then we get `nan` everywhere. Softmax does not change if we subtract the same number from every logit, so we subtract the row maximum first. Every library does this internally.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)    # stability trick, result unchanged
    expZ = np.exp(Z)
    return expZ / expZ.sum(axis=1, keepdims=True)

def one_hot(y, n_classes):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1             # put a 1 in each row's class column
    return Y

def cross_entropy(Y_onehot, P, eps=1e-12):
    return -np.mean(np.sum(Y_onehot * np.log(P + eps), axis=1))

logits = np.array([[2.0, 1.0, 0.1],
                   [1000.0, 1001.0, 1002.0]])    # 2nd row would overflow without the trick
P = softmax(logits)
print("softmax:\n", P)
print("row sums:", P.sum(axis=1))
print("one-hot of [0, 2] with 3 classes:\n", one_hot(np.array([0, 2]), 3))

### 5.2 A general `MLP` class

`layer_sizes = [2, 64, 64, 3]` means 2 inputs, two hidden layers of 64 units, and 3 output classes. Hidden layers use ReLU (or sigmoid), the output uses softmax.

**Weight initialisation.** If the initial weights are too large, the values grow layer after layer and explode; too small, and they shrink to nothing. Both make training fail in deep networks. The fix is to scale the random weights by the number of inputs of the layer:

* **He initialisation** (for ReLU): $W \sim \mathcal{N}\left(0,\ \frac{2}{n_{in}}\right)$
* **Xavier / Glorot initialisation** (for sigmoid and tanh): variance $\frac{1}{n_{in}}$ or $\frac{2}{n_{in}+n_{out}}$

We use He initialisation here. Libraries pick a sensible default for you.

**Mini-batch gradient descent.** There are three ways to choose how much data goes into one update:

| Variant | Examples per update | Pros | Cons |
|---|---|---|---|
| Batch GD | all $N$ | accurate gradient | one update per epoch, slow on big data, needs lots of memory |
| Stochastic GD (SGD) | 1 | many updates, cheap | very noisy, does not use vectorisation |
| **Mini-batch** | e.g. 32 | good balance, uses fast matrix code | one more hyperparameter (batch size) |

Mini-batch is what everyone uses (and confusingly, it is also usually called "SGD"). We **shuffle** the data at the start of every epoch so that batches are different each time; otherwise the network could pick up patterns from the order of the data. One update on one batch is called an **iteration** (or step), so one epoch = $N / \text{batch size}$ iterations.

The backward pass is a loop over the layers that does exactly what we wrote by hand in 4.1.

In [ ]:
ACTIVATIONS = {                      # name -> (function, derivative)
    "relu":    (relu, relu_grad),
    "sigmoid": (sigmoid, sigmoid_grad),
}

class MLP:
    def __init__(self, layer_sizes, activation="relu", seed=0):
        r = np.random.default_rng(seed)
        self.layer_sizes = layer_sizes
        self.W, self.b = [], []
        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.W.append(r.normal(0, np.sqrt(2.0 / n_in), size=(n_in, n_out)))  # He init
            self.b.append(np.zeros(n_out))
        self.act, self.act_grad = ACTIVATIONS[activation]
        self.rng = r

    def forward(self, X):
        self.Zs, self.As = [], [X]          # keep everything for the backward pass
        A = X
        last = len(self.W) - 1
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            Z = A @ W + b
            A = softmax(Z) if i == last else self.act(Z)   # softmax only at the output
            self.Zs.append(Z)
            self.As.append(A)
        return A

    def backward(self, Y_onehot):
        N = Y_onehot.shape[0]
        dZ = (self.As[-1] - Y_onehot) / N   # softmax + cross-entropy gradient
        grads_W, grads_b = [None] * len(self.W), [None] * len(self.W)
        for i in reversed(range(len(self.W))):          # last layer first
            grads_W[i] = self.As[i].T @ dZ
            grads_b[i] = dZ.sum(axis=0)
            if i > 0:
                dA = dZ @ self.W[i].T
                dZ = dA * self.act_grad(self.Zs[i - 1])  # through the activation
        return grads_W, grads_b

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X):
        return self.forward(X).argmax(axis=1)   # class with the highest probability

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

    def fit(self, X, y, epochs=100, lr=0.1, batch_size=32, X_val=None, y_val=None, print_every=0):
        n_classes = self.layer_sizes[-1]
        Y = one_hot(y, n_classes)
        history = {"loss": [], "acc": [], "val_loss": [], "val_acc": []}
        for epoch in range(1, epochs + 1):
            order = self.rng.permutation(len(X))         # shuffle every epoch
            for start in range(0, len(X), batch_size):   # one mini-batch at a time
                idx = order[start:start + batch_size]
                self.forward(X[idx])
                gW, gb = self.backward(Y[idx])
                for i in range(len(self.W)):             # gradient descent step
                    self.W[i] -= lr * gW[i]
                    self.b[i] -= lr * gb[i]
            # record loss/accuracy on the full sets once per epoch
            history["loss"].append(cross_entropy(Y, self.forward(X)))
            history["acc"].append(self.accuracy(X, y))
            if X_val is not None:
                history["val_loss"].append(cross_entropy(one_hot(y_val, n_classes), self.forward(X_val)))
                history["val_acc"].append(self.accuracy(X_val, y_val))
            if print_every and epoch % print_every == 0:
                msg = f"epoch {epoch:4d}  loss {history['loss'][-1]:.4f}  acc {history['acc'][-1]:.3f}"
                if X_val is not None:
                    msg += f"  val_acc {history['val_acc'][-1]:.3f}"
                print(msg)
        return history

Same rule as before: new backprop code gets a gradient check before we trust it. We check a small network with two hidden layers. All relative errors should be tiny.

In [ ]:
small = MLP([4, 5, 6, 3], seed=1)
Xs = rng.normal(size=(8, 4))          # 8 random examples
ys = rng.integers(0, 3, size=8)       # random labels
Ys = one_hot(ys, 3)

small.forward(Xs)
gW, gb = small.backward(Ys)

def small_loss():
    return cross_entropy(Ys, small.forward(Xs))

for i in range(len(small.W)):
    num = numerical_grads(small_loss, {"W": small.W[i], "b": small.b[i]})
    print(f"layer {i}:  W rel. error {rel_error(gW[i], num['W']):.2e}   "
          f"b rel. error {rel_error(gb[i], num['b']):.2e}")

### 5.3 A dataset made from scratch: three spirals

We generate a toy dataset ourselves: three classes of points, each lying on a spiral arm, with some random noise. It is 2D, so we can plot everything, and it is clearly **not** linearly separable.

We make two samples with different seeds: one for training and one for **testing**. A model is only useful if it works on data it has not seen during training (**generalisation**), so we always measure accuracy on separate test data.

In [ ]:
def make_spirals(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    r = np.random.default_rng(seed)
    N = n_per_class * n_classes
    X = np.zeros((N, 2))
    y = np.zeros(N, dtype=int)
    for c in range(n_classes):
        idx = slice(c * n_per_class, (c + 1) * n_per_class)
        radius = np.linspace(0.0, 1.0, n_per_class)                    # from centre outwards
        theta = np.linspace(c * 4, (c + 1) * 4, n_per_class) + r.normal(0, noise, n_per_class)
        X[idx] = np.c_[radius * np.sin(theta), radius * np.cos(theta)]  # polar -> x, y
        y[idx] = c
    return X, y

X_sp, y_sp = make_spirals(seed=0)
X_sp_test, y_sp_test = make_spirals(seed=1)      # a fresh sample for testing
print("X:", X_sp.shape, " y:", y_sp.shape, " classes:", np.unique(y_sp))

plt.figure(figsize=(4, 4))
plt.scatter(X_sp[:, 0], X_sp[:, 1], c=y_sp, cmap="viridis", edgecolors="k", s=25)
plt.title("three spirals"); plt.axis("equal"); plt.show()

### 5.4 Linear model vs. neural network

`MLP([2, 3])` has no hidden layer: it is plain softmax regression (multinomial logistic regression), which can only draw straight boundaries. `MLP([2, 64, 64, 3])` has two hidden layers of 64 ReLU units.

Notice the shape of the learned boundary on the right: it is made of many straight pieces. Each ReLU unit contributes one "bend", and together they approximate the curved spiral arms.

In [ ]:
linear = MLP([2, 3], seed=0)                   # no hidden layer
h_lin = linear.fit(X_sp, y_sp, epochs=300, lr=0.5)

deep = MLP([2, 64, 64, 3], seed=0)             # two hidden layers
h_deep = deep.fit(X_sp, y_sp, epochs=300, lr=0.5, X_val=X_sp_test, y_val=y_sp_test, print_every=50)

print(f"\nlinear model: train acc {linear.accuracy(X_sp, y_sp):.3f}  test acc {linear.accuracy(X_sp_test, y_sp_test):.3f}")
print(f"MLP         : train acc {deep.accuracy(X_sp, y_sp):.3f}  test acc {deep.accuracy(X_sp_test, y_sp_test):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_decision_boundary(linear.predict, X_sp, y_sp, ax=axes[0], title="no hidden layer")
plot_decision_boundary(deep.predict, X_sp, y_sp, ax=axes[1], title="two hidden layers")
axes[2].plot(h_lin["loss"], label="no hidden layer"); axes[2].plot(h_deep["loss"], label="two hidden layers")
axes[2].set_xlabel("epoch"); axes[2].set_ylabel("training loss"); axes[2].legend()
plt.tight_layout(); plt.show()

### 5.5 What the hidden layer size changes

The number of hidden units controls the network's **capacity**: how complicated a function it can represent.

* **Too little capacity** and the network cannot even fit the training data. This is **underfitting** (high bias). Both training and test accuracy are low.
* **Too much capacity** (compared to the amount of data) and the network can start memorising noise in the training set. This is **overfitting** (high variance): training accuracy is high but test accuracy drops. See Section 8.2.

The goal is a model that is complex enough for the real pattern but not so free that it learns the noise.

In [ ]:
sizes = [2, 8, 128]
fig, axes = plt.subplots(1, len(sizes), figsize=(14, 4))
for ax, h in zip(axes, sizes):
    m = MLP([2, h, 3], seed=0)
    m.fit(X_sp, y_sp, epochs=300, lr=0.5)
    acc = m.accuracy(X_sp_test, y_sp_test)
    plot_decision_boundary(m.predict, X_sp, y_sp, ax=ax, title=f"{h} hidden units, test acc {acc:.2f}")
plt.tight_layout(); plt.show()

## 6. A real dataset: handwritten digits

Time for real data. We use the **digits** dataset that ships with scikit-learn: 1,797 grayscale images of handwritten digits, each $8\times8$ pixels with values from 0 (white) to 16 (black). It is a small cousin of the famous MNIST dataset ($28\times28$ pixels, 70,000 images). It is small enough to train our network in seconds and needs no download.

### 6.1 Loading and looking at the data

Always look at your data before training anything. You catch problems early (wrong labels, odd scales, class imbalance) and get a feel for how hard the task is.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_dig, y_dig = digits.data, digits.target       # data: (1797, 64), target: labels 0-9
print("X:", X_dig.shape, " y:", y_dig.shape, " pixel range:", X_dig.min(), "to", X_dig.max())
print("examples per class:", np.bincount(y_dig))   # roughly balanced

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

Each image is $8 \times 8 = 64$ pixels, and `digits.data` already stores every image **flattened** into a vector of 64 numbers. A fully connected network has no idea which pixels were neighbours in the image; for it, the input is just 64 unrelated numbers. Keep this in mind for Section 9.

### 6.2 Train / validation / test split and scaling

We split the data into three parts:

* **Training set** (60%): used to update the weights.
* **Validation set** (20%): used to choose hyperparameters (layer sizes, learning rate, epochs) and to watch for overfitting during training.
* **Test set** (20%): used **once**, at the very end, to report the final result.

Why three sets and not two? If we try many settings and keep the one with the best test score, we have indirectly fitted the test set, and the score we report is too optimistic. The validation set takes that role, so the test set stays an honest estimate of performance on new data. Letting test information influence training in any way is called **data leakage**.

`stratify=y` keeps the class proportions the same in every split, so no split ends up with, say, very few 7s.

**Scaling.** We divide the pixels by 16 so they lie in $[0, 1]$. Neural networks train much better when inputs are small and on similar scales: large inputs give large pre-activations, which saturate sigmoids and make gradients badly balanced. Two common choices:

* **Min-max normalisation**: rescale to $[0, 1]$ (what we do here).
* **Standardisation**: subtract the mean and divide by the standard deviation of each feature. Important: compute the mean and standard deviation on the **training set only**, then apply them to validation and test.

In [ ]:
from sklearn.model_selection import train_test_split

# first split off the test set (20%), then split the rest into train (60%) and validation (20%)
X_tmp, X_test, y_tmp, y_test = train_test_split(X_dig, y_dig, test_size=0.2, stratify=y_dig, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=0)

X_train, X_val, X_test = X_train / 16.0, X_val / 16.0, X_test / 16.0   # scale to [0, 1]

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)

### 6.3 Training our from-scratch MLP

The same `MLP` class we used for the spirals, now with 64 inputs and 10 outputs. Nothing else changes.

**How to read training curves:**

* Training loss should go down steadily. If it goes up or jumps around, the learning rate is probably too high.
* If training and validation curves stay close, the model generalises well.
* If the training loss keeps going down while the validation loss goes up, the model is overfitting.

In [ ]:
scratch_model = MLP([64, 64, 10], seed=0)
hist = scratch_model.fit(X_train, y_train, epochs=60, lr=0.1, batch_size=32,
                         X_val=X_val, y_val=y_val, print_every=10)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist["loss"], label="train"); axes[0].plot(hist["val_loss"], label="validation")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].legend()
axes[1].plot(hist["acc"], label="train"); axes[1].plot(hist["val_acc"], label="validation")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

scratch_test_acc = scratch_model.accuracy(X_test, y_test)
print(f"test accuracy: {scratch_test_acc:.3f}")

### 6.4 Evaluation: accuracy, confusion matrix, precision and recall

**Accuracy** (fraction of correct predictions) is a single number and hides *which* classes get confused. It can also be misleading on imbalanced data: if 95% of emails are not spam, a model that always says "not spam" is 95% accurate and completely useless.

The **confusion matrix** has the true class on the rows and the predicted class on the columns. The diagonal holds the correct predictions; every number off the diagonal is a specific kind of mistake (for example "true 8, predicted 1").

For each class, the classification report gives:

* **Precision** = of everything predicted as this class, how much really was. $\frac{TP}{TP + FP}$
* **Recall** = of everything that really is this class, how much we found. $\frac{TP}{TP + FN}$
* **F1-score** = harmonic mean of precision and recall, a single number that is high only if both are high.
* **Support** = number of test examples of that class.

($TP$, $FP$, $FN$: true positives, false positives, false negatives.)

Finally, look at the misclassified images. It is often the quickest way to understand what a model finds hard, and some of the "mistakes" turn out to be badly written digits that a person would also get wrong.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred_scratch = scratch_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_scratch)
ConfusionMatrixDisplay(cm).plot(cmap="Blues", colorbar=False)
plt.title("MLP on digits (test set)"); plt.show()

print(classification_report(y_test, y_pred_scratch, digits=3))

# show some of the mistakes
wrong = np.where(y_pred_scratch != y_test)[0]
print(f"{len(wrong)} mistakes out of {len(y_test)}")
fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(12, 2))
for ax, i in zip(np.atleast_1d(axes), wrong[:8]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[i]}, pred {y_pred_scratch[i]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Neural networks from libraries

We now know every piece of what happens inside a network. In practice nobody writes backprop by hand. Libraries:

* compute gradients automatically,
* run on GPUs,
* provide ready-made layers, losses and better optimisers,
* are tested by thousands of users, so they have far fewer bugs than our code.

Here is the same digits problem in three popular libraries.

### 7.1 scikit-learn: `MLPClassifier`

The shortest route. Its main parameters:

* `hidden_layer_sizes=(64,)`: one hidden layer of 64 units. `(128, 64)` would be two hidden layers.
* `activation`: `"relu"`, `"tanh"`, `"logistic"` (sigmoid) or `"identity"`.
* `solver`: the optimiser. `"adam"` is a good default, `"sgd"` is plain mini-batch gradient descent, `"lbfgs"` works well on small datasets.
* `alpha`: strength of L2 regularisation (Section 8.2).
* `max_iter`: maximum number of epochs.
* `early_stopping=True` sets aside part of the training data and stops when the validation score stops improving.

The output layer and loss (softmax + cross-entropy) are chosen automatically.

**Limitations:** CPU only, so not suited to large datasets, and only fully connected layers. Good for small tabular problems and quick baselines.

In [ ]:
from sklearn.neural_network import MLPClassifier

sk_model = MLPClassifier(hidden_layer_sizes=(64,),   # 64 -> 64 -> 10, same as ours
                         activation="relu",
                         solver="adam",
                         learning_rate_init=1e-3,
                         max_iter=1000,
                         random_state=0)
sk_model.fit(X_train, y_train)                       # the whole training loop
sk_test_acc = sk_model.score(X_test, y_test)         # accuracy

print(f"test accuracy: {sk_test_acc:.3f}")
print("weight shapes:", [W.shape for W in sk_model.coefs_])   # same shapes as our self.W

### 7.2 PyTorch

PyTorch is the most widely used library in research. Its main pieces:

* **Tensor**: PyTorch's version of a NumPy array. It can live on a GPU and can remember how it was computed.
* **Autograd**: PyTorch records every operation done during the forward pass (the **computational graph**). Calling `loss.backward()` walks that graph backwards and fills in `.grad` for every parameter. This is exactly the backpropagation we wrote by hand, done automatically for any model.
* **`nn.Module`** / **`nn.Sequential`**: containers for layers. `nn.Linear(64, 64)` is one fully connected layer, a weight matrix plus a bias, like one entry of our `self.W` and `self.b`.
* **Loss**: `nn.CrossEntropyLoss` takes the raw **logits** and applies softmax internally, in a numerically safer way. So the model has **no softmax at the end**. (Adding one anyway is a common bug: training still runs, but worse.)
* **Optimiser**: `torch.optim.SGD`, `torch.optim.Adam`, ... It performs the update step `w -= lr * grad` (or something smarter) for all parameters.
* **`DataLoader`**: cuts the dataset into shuffled mini-batches for us.

First, to convince ourselves that autograd is not magic, we give PyTorch the exact weights of our small NumPy network from Section 5.2 and compare the gradients.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

print("torch", torch.__version__)

# Copy the weights of `small` (MLP([4, 5, 6, 3]) from Section 5.2) into tensors
Wt = [torch.tensor(W, requires_grad=True) for W in small.W]   # requires_grad: track gradients
bt = [torch.tensor(b, requires_grad=True) for b in small.b]

# Forward pass, same as our MLP
A = torch.tensor(Xs)
for i in range(len(Wt)):
    A = A @ Wt[i] + bt[i]
    if i < len(Wt) - 1:
        A = torch.relu(A)
loss = F.cross_entropy(A, torch.tensor(ys))     # softmax + cross-entropy in one call
loss.backward()                                 # autograd computes all gradients

# Our hand-written backward pass
small.forward(Xs)
gW, gb = small.backward(Ys)
for i in range(len(Wt)):
    diff = np.abs(Wt[i].grad.numpy() - gW[i]).max()
    print(f"layer {i}: largest difference between PyTorch and our dW = {diff:.2e}")

The differences are at the level of floating-point rounding: our backprop and PyTorch compute the same thing.

Now the real model. Two details in the training function below:

* `model.train()` and `model.eval()` switch the model between training and evaluation mode. Our model behaves the same in both, but layers like dropout (Section 8.2) behave differently, so it is a good habit.
* `torch.no_grad()` tells PyTorch not to record the graph during evaluation, which saves memory and time.

We use the **Adam** optimiser instead of plain SGD. Adam keeps a running average of each weight's past gradients (**momentum**) and of their size, and uses them to give every weight its own step size. It is usually much less sensitive to the learning rate than plain SGD, and `lr=1e-3` is a good starting point.

In [ ]:
def torch_accuracy(model, X, y):
    model.eval()                                   # evaluation mode
    with torch.no_grad():                          # no gradients needed here
        preds = model(torch.tensor(X, dtype=torch.float32)).argmax(dim=1).numpy()
    return np.mean(preds == y)

def train_torch_model(model, X_tr, y_tr, X_va, y_va, epochs=50, lr=1e-3, batch_size=32, seed=0, print_every=10):
    torch.manual_seed(seed)
    dataset = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                            torch.tensor(y_tr, dtype=torch.long))      # labels as integers
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)  # mini-batches
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {"loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()                       # training mode
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()           # clear old gradients (PyTorch adds them up otherwise)
            loss = loss_fn(model(xb), yb)   # forward + loss
            loss.backward()                 # backward (autograd)
            optimizer.step()                # update the weights
            total += loss.item() * len(xb)
        history["loss"].append(total / len(dataset))
        history["val_acc"].append(torch_accuracy(model, X_va, y_va))
        if print_every and epoch % print_every == 0:
            print(f"epoch {epoch:3d}  loss {history['loss'][-1]:.4f}  val_acc {history['val_acc'][-1]:.3f}")
    return history

torch.manual_seed(0)
torch_model = nn.Sequential(
    nn.Linear(64, 64),   # input -> hidden
    nn.ReLU(),
    nn.Linear(64, 10),   # hidden -> output logits (no softmax here)
)
print(torch_model)
print("number of parameters:", sum(p.numel() for p in torch_model.parameters()))

torch_hist = train_torch_model(torch_model, X_train, y_train, X_val, y_val, epochs=50)
torch_test_acc = torch_accuracy(torch_model, X_test, y_test)
print(f"PyTorch test accuracy: {torch_test_acc:.3f}")

### 7.3 Keras

Keras sits between the two. You describe the layers, and the training loop is written for you:

* `keras.Sequential([...])`: a stack of layers. `Dense` is Keras' name for a fully connected layer.
* `compile(...)`: choose the optimiser, the loss and the metrics to report.
* `fit(...)`: run the training loop (epochs, batches, validation).
* `evaluate(...)`: compute loss and metrics on new data.
* `summary()`: print the layers and their parameter counts.

Keras 3 can run on top of TensorFlow, PyTorch or JAX (the **backend**). The cell below uses TensorFlow if it is installed and falls back to PyTorch otherwise. If Keras is not installed at all, the section is skipped.

Here the model *does* end with a softmax, because Keras' `sparse_categorical_crossentropy` expects probabilities by default. "Sparse" means the labels are plain integers like `3`, not one-hot vectors.

In [ ]:
import os, importlib.util
# Keras needs a backend; use PyTorch if TensorFlow is not installed
if importlib.util.find_spec("tensorflow") is None:
    os.environ.setdefault("KERAS_BACKEND", "torch")

try:
    import keras
    KERAS_OK = True
    print("keras", keras.__version__, "backend:", keras.backend.backend())
except ImportError:
    KERAS_OK = False
    print("Keras not installed, skipping this part (pip install keras to try it).")

In [ ]:
keras_test_acc = None
if KERAS_OK:
    keras.utils.set_random_seed(0)
    keras_model = keras.Sequential([
        keras.Input(shape=(64,)),                        # 64 pixels
        keras.layers.Dense(64, activation="relu"),       # hidden layer
        keras.layers.Dense(10, activation="softmax"),    # output probabilities
    ])
    keras_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                        loss="sparse_categorical_crossentropy",
                        metrics=["accuracy"])
    keras_model.summary()
    keras_hist = keras_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                 epochs=50, batch_size=32, verbose=0)   # verbose=0: no progress bar
    keras_test_acc = keras_model.evaluate(X_test, y_test, verbose=0)[1]  # [loss, accuracy]
    print(f"Keras test accuracy: {keras_test_acc:.3f}")

### 7.4 Comparing all of them

All four models have the same architecture (64 → 64 → 10). Differences in accuracy come from the optimiser (plain SGD for ours, Adam for the others), the number of epochs and the random initialisation, not from anything fundamental. With a dataset this small, differences of one or two test examples change the accuracy by about 0.3%, so do not read too much into small gaps.

In [ ]:
results = {"Scratch": scratch_test_acc, "scikit-learn": sk_test_acc, "PyTorch": torch_test_acc}
if keras_test_acc is not None:
    results["Keras"] = keras_test_acc

print(f"{'model':15s} test accuracy")
for name, acc in results.items():
    print(f"{name:15s} {acc:.3f}")

plt.figure(figsize=(6, 3))
plt.barh(list(results), list(results.values()))
plt.xlim(0.9, 1.0); plt.xlabel("test accuracy"); plt.show()

| | Scratch | scikit-learn | PyTorch | Keras |
|---|---|---|---|---|
| Lines of code for the model | many | 2 | ~5 + loop | ~10 |
| You write backprop | yes | no | no | no |
| You write the training loop | yes | no | yes | no |
| GPU | no | no | yes | yes |
| Custom architectures | painful | no | yes | yes |
| Good for | understanding | quick baselines | research, full control | fast prototyping |

## 8. Practical notes

### 8.1 The learning rate

The learning rate is usually the single most important hyperparameter. The experiment below trains the same network on the spirals with four learning rates:

* **too small** (0.001): the loss barely moves; training would take forever,
* **reasonable** (0.05, 0.5): the loss goes down steadily,
* **too large** (5.0): the updates overshoot, the loss jumps around or blows up to `nan`.

A common practical approach is to try values spaced by factors of 10 (0.0001, 0.001, 0.01, 0.1, 1) and keep the best on the validation set. Many people also **decrease the learning rate during training** (a learning rate *schedule*): big steps early on, small careful steps near the end.

In [ ]:
plt.figure(figsize=(6, 3.5))
for lr in [0.001, 0.05, 0.5, 5.0]:
    with np.errstate(all="ignore"):   # the largest lr overflows; that is the point
        h = MLP([2, 64, 64, 3], seed=0).fit(X_sp, y_sp, epochs=150, lr=lr)
    print(f"lr = {lr:<6} final training loss = {h['loss'][-1]:.4f}")
    plt.plot(h["loss"], label=f"lr = {lr}")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.show()

### 8.2 Overfitting

A network with many parameters can memorise its training set instead of learning the general pattern. To see it happen, we train a fairly large network (two hidden layers of 256 units, about 85,000 parameters) on only **100** training digits and watch both curves.

Training accuracy quickly reaches 100% and training loss keeps falling, but validation accuracy stalls far below, and validation loss starts to **rise** again. From that point on, every extra epoch makes the model worse on new data.

**Ways to fight overfitting:**

* **More data**, or **data augmentation** (for images: small shifts, rotations, noise).
* **A smaller network**, with fewer parameters to memorise with.
* **Early stopping**: monitor the validation loss and keep the weights from the epoch where it was lowest.
* **L2 regularisation (weight decay)**: add $\lambda \sum w^2$ to the loss, which pushes weights towards small values and gives smoother functions. `alpha=` in scikit-learn, `weight_decay=` in PyTorch optimisers.
* **Dropout**: during training, randomly switch off a fraction of the units (for example 50%) at every step, so the network cannot rely on any single unit. `nn.Dropout(p=0.5)` in PyTorch. It is turned off at evaluation time, which is why `model.eval()` matters.

In [ ]:
# only 100 training examples, a big network, many epochs
X_small, y_small = X_train[:100], y_train[:100]
overfit_model = MLP([64, 256, 256, 10], seed=0)
h_over = overfit_model.fit(X_small, y_small, epochs=300, lr=0.1, batch_size=16, X_val=X_val, y_val=y_val)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(h_over["loss"], label="train"); axes[0].plot(h_over["val_loss"], label="validation")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].legend(); axes[0].set_title("loss")
axes[1].plot(h_over["acc"], label="train"); axes[1].plot(h_over["val_acc"], label="validation")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(); axes[1].set_title("accuracy")
plt.tight_layout(); plt.show()

best = int(np.argmin(h_over["val_loss"]))
print(f"final train acc {h_over['acc'][-1]:.3f}, final val acc {h_over['val_acc'][-1]:.3f}")
print(f"validation loss was lowest at epoch {best + 1}; early stopping would keep those weights")

### 8.3 Other things worth knowing

* **Always scale your inputs** (Section 6.2). Unscaled features with very different ranges make training slow and unstable.
* **Start simple.** Begin with a small network and a known-good learning rate, check that the training loss goes down, and only then make things bigger. A useful debugging trick: make sure your network can overfit a tiny dataset (say 10 examples). If it cannot even do that, there is a bug.
* **Batch size.** Larger batches give smoother gradients and use the hardware better, but need more memory and sometimes generalise slightly worse. 32 to 256 are common values.
* **Batch normalisation** (`nn.BatchNorm1d`) normalises the values inside the network after each layer. It often makes deep networks train faster and less sensitive to initialisation.
* **Fully connected networks are not the best tool for every job.** For images there are CNNs (next week), for sequences recurrent networks and transformers, and for small tabular datasets tree-based models like gradient boosting are often hard to beat.

## 9. Preview of next week: Convolutional Neural Networks

The networks in this lab are **fully connected**: the image is flattened into a vector and every pixel is connected to every hidden unit. For images this has two big problems:

1. **Too many parameters.** A $224\times224$ colour image has $224 \times 224 \times 3 = 150{,}528$ inputs. One hidden layer of 1,000 units already needs about 150 million weights.
2. **No notion of space.** If you shuffle the pixels in the same way in every image, a fully connected network learns exactly as well as before. It does not know that neighbouring pixels belong together, and a pattern learned in one corner of the image has to be learned again for every other position.

A **convolutional layer** fixes both. Instead of one weight per pixel, it slides a small **filter** (also called a kernel, for example $3\times3$) across the image and computes a weighted sum at every position. The same 9 weights are reused at every position (**weight sharing**), and each output only looks at a small neighbourhood (**local connectivity**). The output is called a **feature map**.

Below is a hand-made vertical-edge filter applied to one digit: red and blue show where the image changes from light to dark from left to right, or the other way round. Next week, the network will **learn** its filters from data.

In [ ]:
def conv2d(image, kernel):
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for i in range(out.shape[0]):              # slide the filter over every position
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i + kh, j:j + kw] * kernel)
    return out

vertical_edge = np.array([[1, 0, -1],
                          [1, 0, -1],
                          [1, 0, -1]], dtype=float)
img = digits.images[0]
edges = conv2d(img, vertical_edge)

fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("input 8x8")
axes[1].imshow(vertical_edge, cmap="bwr"); axes[1].set_title("3x3 filter")
axes[2].imshow(edges, cmap="bwr"); axes[2].set_title("feature map 6x6")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print("weights in this filter:", vertical_edge.size,
      "   weights in a dense layer from 64 pixels to 36 outputs:", 64 * 36)

## 10. Further reading

* Russell and Norvig, *Artificial Intelligence: A Modern Approach* (4th ed.), Chapter 21: Deep Learning.
* Michael Nielsen, *Neural Networks and Deep Learning*, a free online book. Chapters 1 and 2 explain neurons and backpropagation very clearly: http://neuralnetworksanddeeplearning.com
* Goodfellow, Bengio and Courville, *Deep Learning*, MIT Press, Chapters 6 to 8 (free online): https://www.deeplearningbook.org
* 3Blue1Brown, *Neural Networks* video series on YouTube, for visual intuition about gradient descent and backpropagation.
* Stanford CS231n course notes, "Neural Networks Part 1 to 3" and "Backpropagation, Intuitions": https://cs231n.github.io
* TensorFlow Playground, train small networks in the browser and watch the decision boundary change: https://playground.tensorflow.org
* PyTorch tutorial, "Learn the Basics": https://pytorch.org/tutorials/beginner/basics/intro.html
* scikit-learn user guide, "Neural network models (supervised)": https://scikit-learn.org/stable/modules/neural_networks_supervised.html

## 11. Tasks

Make sure you have run Sections 0 to 9 first; the tasks use functions defined there (`relu`, `sigmoid`, `MLP`, `train_torch_model`, ...).

**How the tasks work**

* Each task has one (or two) cells for your code. Replace the `raise NotImplementedError(...)` line or the `None` placeholder with your solution.
* Right after it comes a **check cell** with `assert` statements. Run it. If it prints `Task N passed`, your solution is correct. If it stops with an `AssertionError`, the message tells you what is wrong; fix your code, then run both cells again.
* Do not edit the check cells.
* The tasks are independent of each other, so if you get stuck on one you can move on to the next.

### Task 1: tanh and leaky ReLU

*Related section: 1.2*

Write four functions. They must work element-wise on NumPy arrays of any shape.

1. `tanh(z)` using the formula $\dfrac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$. Use `np.exp`, **not** `np.tanh` (the check compares against `np.tanh`).
2. `tanh_grad(z)`: the derivative of tanh, $1 - \tanh^2(z)$.
3. `leaky_relu(z, alpha=0.01)`: returns `z` where `z > 0` and `alpha * z` elsewhere.
4. `leaky_relu_grad(z, alpha=0.01)`: returns `1` where `z > 0` and `alpha` elsewhere.

Hint: `np.where(condition, value_if_true, value_if_false)` works element-wise.

In [ ]:
# TASK 1
def tanh(z):
    # YOUR CODE HERE
    raise NotImplementedError("Task 1: write tanh")

def tanh_grad(z):
    # YOUR CODE HERE
    raise NotImplementedError("Task 1: write tanh_grad")

def leaky_relu(z, alpha=0.01):
    # YOUR CODE HERE
    raise NotImplementedError("Task 1: write leaky_relu")

def leaky_relu_grad(z, alpha=0.01):
    # YOUR CODE HERE
    raise NotImplementedError("Task 1: write leaky_relu_grad")

In [ ]:
# CHECK TASK 1 (do not edit)
_z = np.linspace(-4, 4, 41)
assert tanh(np.array(0.0)) == 0, "tanh(0) should be 0"
assert np.allclose(tanh(_z), np.tanh(_z)), "tanh does not match np.tanh"
_num = (tanh(_z + 1e-5) - tanh(_z - 1e-5)) / 2e-5
assert np.allclose(tanh_grad(_z), _num, atol=1e-6), "tanh_grad does not match the numerical slope of tanh"
assert np.allclose(leaky_relu(np.array([-10., -1., 0., 2.])), [-0.1, -0.01, 0., 2.]), "leaky_relu values are wrong"
assert np.allclose(leaky_relu(np.array([-10.]), alpha=0.2), [-2.]), "leaky_relu ignores alpha"
assert np.allclose(leaky_relu_grad(np.array([-3., 4.])), [0.01, 1.]), "leaky_relu_grad values are wrong"
assert tanh(np.ones((3, 2))).shape == (3, 2), "tanh should keep the input shape"
print("Task 1 passed")

### Task 2: forward pass of a 3-4-2 network

*Related sections: 2.1 and 2.2*

Write `forward_3_4_2(X, W1, b1, W2, b2)` for a network with 3 inputs, a hidden layer of 4 units with **ReLU**, and 2 outputs with **sigmoid**.

It must return a tuple `(H, Y_hat)`: the hidden activations (shape `(n, 4)`) and the outputs (shape `(n, 2)`). Use `relu` and `sigmoid` from Section 1.2 and matrix products, no loops.

Before running the check, try computing the hidden layer for the first example by hand.

In [ ]:
# Fixed inputs and weights for Task 2 (do not change)
X_t2  = np.array([[1.0, 0.0,  2.0],
                  [0.0, 1.0, -1.0]])
W1_t2 = np.array([[1.0, -1.0,  0.0,  2.0],
                  [0.0,  1.0,  1.0, -1.0],
                  [1.0,  0.0, -1.0,  1.0]])
b1_t2 = np.array([0.0, 1.0, 0.0, -1.0])
W2_t2 = np.array([[ 0.5, -0.5],
                  [ 1.0,  0.0],
                  [-1.0,  1.0],
                  [ 0.0,  0.25]])
b2_t2 = np.array([0.0, -1.0])

In [ ]:
# TASK 2
def forward_3_4_2(X, W1, b1, W2, b2):
    # YOUR CODE HERE
    raise NotImplementedError("Task 2: write the forward pass")

In [ ]:
# CHECK TASK 2 (do not edit)
_out = forward_3_4_2(X_t2, W1_t2, b1_t2, W2_t2, b2_t2)
assert isinstance(_out, tuple) and len(_out) == 2, "return a tuple (H, Y_hat)"
_H, _Y = _out
assert _H.shape == (2, 4), f"H should have shape (2, 4), got {_H.shape}"
assert _Y.shape == (2, 2), f"Y_hat should have shape (2, 2), got {_Y.shape}"
assert np.all(_H >= 0), "hidden layer uses ReLU, so H cannot be negative"
assert np.allclose(_H, [[3, 0, 0, 3], [0, 2, 2, 0]]), "hidden activations are wrong (did you add b1 and apply ReLU?)"
assert np.allclose(_Y, 1 / (1 + np.exp(-np.array([[1.5, -1.75], [0.0, 1.0]])))), "outputs are wrong (did you add b2 and apply sigmoid?)"
print("Task 2 passed")

### Task 3: mean squared error and its gradient

*Related section: 3.1*

Write two functions:

1. `mse(y_true, y_pred)`: $\frac{1}{N}\sum (\hat y_i - y_i)^2$. It should work for arrays of any shape, so $N$ is the **total number of elements** (`y_true.size`), not just the number of rows.
2. `mse_grad(y_true, y_pred)`: the derivative of the MSE with respect to `y_pred`. It returns an array with the same shape as `y_pred`.

Work out the derivative on paper first: differentiate $\frac{1}{N}\sum (\hat y_i - y_i)^2$ with respect to one $\hat y_k$.

In [ ]:
# TASK 3
def mse(y_true, y_pred):
    # YOUR CODE HERE
    raise NotImplementedError("Task 3: write mse")

def mse_grad(y_true, y_pred):
    # YOUR CODE HERE
    raise NotImplementedError("Task 3: write mse_grad")

In [ ]:
# CHECK TASK 3 (do not edit)
_yt = np.array([1.0, 2.0, 3.0, 4.0])
_yp = np.array([1.5, 2.0, 2.0, 5.0])
assert np.isclose(mse(_yt, _yp), 0.5625), "mse value is wrong for the 1D example"
assert np.isclose(mse(_yt, _yt), 0.0), "mse of a perfect prediction must be 0"
_yt2, _yp2 = rng.normal(size=(4, 3)), rng.normal(size=(4, 3))
assert np.isclose(mse(_yt2, _yp2), np.mean((_yt2 - _yp2) ** 2)), "mse should average over all elements"
_g = mse_grad(_yt2, _yp2)
assert _g.shape == _yp2.shape, "mse_grad must have the same shape as y_pred"
_num = np.zeros_like(_yp2)
for _idx in np.ndindex(_yp2.shape):
    _p, _m = _yp2.copy(), _yp2.copy()
    _p[_idx] += 1e-6; _m[_idx] -= 1e-6
    _num[_idx] = (mse(_yt2, _p) - mse(_yt2, _m)) / 2e-6
assert np.allclose(_g, _num, atol=1e-6), "mse_grad does not match the numerical gradient"
print("Task 3 passed")

### Task 4: backprop with a tanh hidden layer

*Related sections: 4.1 to 4.3*

Below is a forward pass for the same network as in Section 4.2, except that the hidden layer uses **tanh** instead of sigmoid. The output is still sigmoid and the loss is still BCE.

Write `backward_tanh(params, cache, Y)`. It returns a dictionary with keys `"W1"`, `"b1"`, `"W2"`, `"b2"`, like `backward` in Section 4.2. Only one line is really different: the derivative of the hidden activation. Remember that $\tanh'(z) = 1 - \tanh^2(z)$, and `cache["A1"]` already holds $\tanh(Z_1)$.

The check runs a gradient check on your function (using `numerical_grads` from 4.3), then trains a network on XOR with your gradients.

In [ ]:
# Forward pass with a tanh hidden layer (given, do not change)
def forward_tanh(params, X):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = np.tanh(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = sigmoid(Z2)
    return A2, {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}

In [ ]:
# TASK 4
def backward_tanh(params, cache, Y):
    # YOUR CODE HERE
    raise NotImplementedError("Task 4: write backward_tanh")

In [ ]:
# CHECK TASK 4 (do not edit)
_params = init_params(2, 5, 1, seed=3)
_X = rng.normal(size=(7, 2)); _Y = rng.integers(0, 2, size=(7, 1)).astype(float)
_A2, _cache = forward_tanh(_params, _X)
_g = backward_tanh(_params, _cache, _Y)
assert isinstance(_g, dict) and set(_g) == {"W1", "b1", "W2", "b2"}, "return a dict with keys W1, b1, W2, b2"
_ng = numerical_grads(lambda: bce(_Y, forward_tanh(_params, _X)[0]), _params)
for _k in _params:
    assert _g[_k].shape == _params[_k].shape, f"grad of {_k} has shape {_g[_k].shape}, expected {_params[_k].shape}"
    assert rel_error(_g[_k], _ng[_k]) < 1e-6, f"gradient of {_k} fails the gradient check"
_params = init_params(2, 4, 1, seed=1)
for _ in range(3000):
    _A2, _cache = forward_tanh(_params, X_bool)
    _g = backward_tanh(_params, _cache, Y_xor)
    for _k in _params:
        _params[_k] -= 0.5 * _g[_k]
assert np.array_equal((forward_tanh(_params, X_bool)[0] > 0.5).astype(int).ravel(), y_xor), "training with your gradients did not solve XOR"
print("Task 4 passed")

### Task 5: count the parameters of a network

*Related sections: 2.2 and 5.2*

Write `count_params(layer_sizes)` that returns the total number of trainable parameters (weights **and** biases) of a fully connected network with the given layer sizes.

For example `[2, 3]` has a $2\times3$ weight matrix and 3 biases, so 9 parameters. Use a formula; do not build an `MLP`.

Then use it to check the claim in Section 9: how many parameters does `[224*224*3, 1000]` have?

In [ ]:
# TASK 5
def count_params(layer_sizes):
    # YOUR CODE HERE
    raise NotImplementedError("Task 5: write count_params")

In [ ]:
# CHECK TASK 5 (do not edit)
assert count_params([2, 3]) == 9, "[2, 3] should have 9 parameters"
assert count_params([2, 64, 3]) == 387, "[2, 64, 3] should have 387 parameters"
for _sizes in ([2, 64, 64, 3], [64, 32, 10], [784, 128, 64, 10]):
    _m = MLP(_sizes)
    _true = sum(W.size for W in _m.W) + sum(b.size for b in _m.b)
    assert count_params(_sizes) == _true, f"wrong count for {_sizes}: expected {_true}"
assert isinstance(count_params([5, 1]), (int, np.integer)), "return an integer"
print("Task 5 passed")

### Task 6: build a "rings" dataset and train a network on it

*Related sections: 5.2 to 5.4*

**Part A.** Write `make_rings(n_per_class=200, noise=0.1, seed=0)` that returns `(X, y)`:

* class 0: points on a circle of radius **1**, class 1: points on a circle of radius **2** (both centred at the origin),
* draw each point's angle uniformly from $[0, 2\pi)$, and add Gaussian noise with standard deviation `noise` to the radius,
* `X` has shape `(2 * n_per_class, 2)`, `y` has shape `(2 * n_per_class,)` with integer labels 0 and 1,
* use `np.random.default_rng(seed)` so that the same seed gives the same data.

A point with angle $\theta$ and radius $r$ is $(r\cos\theta,\ r\sin\theta)$. Look at `make_spirals` in Section 5.3 for inspiration.

**Part B.** Create an `MLP` called `ring_model` and train it on `make_rings(seed=0)` so that it reaches **at least 95% accuracy** on the test set `make_rings(seed=123)`. Choose the layer sizes, learning rate and number of epochs yourself. Plotting the decision boundary with `plot_decision_boundary` is a good way to see what your model learned.

In [ ]:
# TASK 6, part A
def make_rings(n_per_class=200, noise=0.1, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError("Task 6: write make_rings")

In [ ]:
# TASK 6, part B
X_ring, y_ring = make_rings(seed=0)
X_ring_test, y_ring_test = make_rings(seed=123)

ring_model = None   # YOUR CODE HERE: create the MLP and call .fit(...)

In [ ]:
# CHECK TASK 6 (do not edit)
_X, _y = make_rings(n_per_class=150, noise=0.05, seed=7)
assert _X.shape == (300, 2), f"X should have shape (300, 2), got {_X.shape}"
assert _y.shape == (300,), f"y should have shape (300,), got {_y.shape}"
assert set(np.unique(_y)) == {0, 1} and np.sum(_y == 0) == 150, "need 150 points of each class"
_r = np.linalg.norm(_X, axis=1)
assert abs(_r[_y == 0].mean() - 1) < 0.05, "class 0 should sit at radius 1"
assert abs(_r[_y == 1].mean() - 2) < 0.05, "class 1 should sit at radius 2"
assert np.array_equal(make_rings(seed=3)[0], make_rings(seed=3)[0]), "same seed must give the same data"
assert not np.array_equal(make_rings(seed=3)[0], make_rings(seed=4)[0]), "different seeds should give different data"
_ang = np.arctan2(_X[:, 1], _X[:, 0])
assert _ang.min() < -2.5 and _ang.max() > 2.5, "angles should cover the whole circle"
assert ring_model is not None, "create and train ring_model in part B"
assert isinstance(ring_model, MLP), "ring_model should be an MLP from Section 5.2"
_acc = ring_model.accuracy(X_ring_test, y_ring_test)
assert _acc >= 0.95, f"test accuracy is {_acc:.3f}, you need at least 0.95"
print(f"Task 6 passed (test accuracy {_acc:.3f})")

### Task 7: your own PyTorch network on the digits

*Related section: 7.2*

Build a PyTorch model called `digit_model` with **at least two hidden layers** and train it with `train_torch_model` (from 7.2) on `X_train, y_train`, validating on `X_val, y_val`. It must reach **at least 96% test accuracy**.

Things you can change: number of hidden layers and units, activation (`nn.ReLU`, `nn.Tanh`, `nn.LeakyReLU`), learning rate, number of epochs, batch size. Use the validation accuracy to make your choices, not the test accuracy (Section 6.2 explains why).

Remember: no softmax at the end, `nn.CrossEntropyLoss` takes care of it.

In [ ]:
# TASK 7
torch.manual_seed(0)
digit_model = None   # YOUR CODE HERE: build an nn.Sequential with at least two hidden layers

# YOUR CODE HERE: train it with train_torch_model(...)

In [ ]:
# CHECK TASK 7 (do not edit)
assert digit_model is not None, "build digit_model first"
assert isinstance(digit_model, nn.Module), "digit_model must be a PyTorch nn.Module"
_linear = [m for m in digit_model.modules() if isinstance(m, nn.Linear)]
assert len(_linear) >= 3, f"need at least two hidden layers (3 nn.Linear layers), found {len(_linear)} Linear layer(s)"
assert _linear[0].in_features == 64 and _linear[-1].out_features == 10, "input must be 64 features and output 10 classes"
_acc = torch_accuracy(digit_model, X_test, y_test)
assert _acc >= 0.96, f"test accuracy is {_acc:.3f}, you need at least 0.96 (did you train it?)"
print(f"Task 7 passed (test accuracy {_acc:.3f})")